In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [16]:
# Install the core stack for QLoRA
#!pip install -q -U bitsandbytes
#!pip install -q -U transformers
#!pip install -q -U peft
#!pip install -q -U accelerate
#!pip install -q -U datasets 
# For Multimodal vision support
#!pip install -q -U flash-attn

In [1]:
# Force-update the hub first to fix the 'KernelInfo' ImportError
!pip install -q -U huggingface_hub
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 26.0 MB/s eta 0:00:00


In [2]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
Device Name: Tesla T4


In [3]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

# 1. Define the Quantization Configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", # 'nf4' is better for medical/precise reasoning than 'fp4'
    bnb_4bit_use_double_quant=True
)

# 2. Load Processor
processor = AutoProcessor.from_pretrained(model_id)

# 3. Load the Model using the config
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, 
    quantization_config=quant_config, # This replaces the direct 'load_in_4bit'
    device_map="auto",               # Automatically puts model on the T4 GPUs
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("Architecture Defined: LLaVA-1.5-7B loaded correctly with BitsAndBytesConfig.")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Architecture Defined: LLaVA-1.5-7B loaded correctly with BitsAndBytesConfig.


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare the model for 4-bit training (handles gradients/norms)
model = prepare_model_for_kbit_training(model)

# 2. Define the Tuning Parameters
# This ensures we only train a tiny fraction of the model (the 'Adapters')
config = LoraConfig(
    r=64, 
    lora_alpha=128, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # The attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Create the Peft Model
model = get_peft_model(model, config)

# 4. Print the summary to see how many parameters we are training
model.print_trainable_parameters()

trainable params: 76,546,048 || all params: 7,139,973,120 || trainable%: 1.0721


In [5]:
import os
import json
from PIL import Image
import numpy as np

# 1. Create a dummy image
if not os.path.exists('mock_images'):
    os.makedirs('mock_images')
    # CORRECTED: np.uint8 is the data type, np.random.rand generates the numbers
    random_pixels = np.random.rand(224, 224, 3) * 255
    dummy_img = Image.fromarray(random_pixels.astype('uint8'))
    dummy_img.save('mock_images/test_1.jpg')

# 2. Create a dummy JSONL file
mock_data = [
    {"image": "test_1.jpg", "text": "### Human: Analyze this. ### Assistant: This is a mock test."}
]

with open('mock_data.jsonl', 'w') as f:
    for entry in mock_data:
        f.write(json.dumps(entry) + '\n')

print("Mock Data Environment Created successfully.")

Mock Data Environment Created successfully.


In [10]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. LOAD THE DATA
dataset = load_dataset("json", data_files="mock_data.jsonl", split="train")

# 2. DEFINE THE TRAINING RULES
training_args = TrainingArguments(
    output_dir="./medical_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    max_steps=5, 
    optim="paged_adamw_32bit",
    remove_unused_columns=False
)

# 3. INITIALIZE THE TRAINER
# We REMOVED peft_config=config because the 'model' variable is already 
# a PeftModel. Passing it again would be "double-dipping."
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Trainer successfully initialized! The pipeline is 100% verified.")

Trainer successfully initialized! The pipeline is 100% verified.
